## Step 3: Data Science & Machine Learning

**Project:** AI-Powered Sales Data Insights with Snowflake + Cortex Analyst

**Architecture:** Ingestion → Transformation → DELIVERY (ML Layer) → Observability

**5 ML Techniques:**

| # | Technique | Method | Business Value |
|---|-----------|--------|----------------|
| 1 | Sales Forecasting | Exponential Smoothing | Predict next 7 days revenue |
| 2 | Customer Segmentation | RFM Analysis | Target marketing spend |
| 3 | Anomaly Detection | Z-Score Analysis | Flag unusual sales days |
| 4 | Product Classification | BCG Matrix | Portfolio optimization |
| 5 | Territory Scoring | Composite Weighted Score | Investment priorities |

**Prerequisite:** Step 2 must be completed.

In [ ]:
# Snowpark Pandas API
import modin.pandas as spd
import snowflake.snowpark.modin.plugin

# Snowpark DataFrame API
import snowflake.snowpark.functions as F
from snowflake.snowpark.context import get_active_session

# Standard libraries for ML
import pandas as pd
import numpy as np

In [ ]:
session = get_active_session()
session.query_tag = {
    "origin": "raj_manala",
    "name": "sales_analytics_ai_insights",
    "version": {"major": 1, "minor": 0},
    "attributes": {"step": "data_science_ml"}
}
print(f"Session ready: {session.get_current_database()}")

In [ ]:
%%sql -r dataframe_1
USE DATABASE SALES_ANALYTICS_2025;
USE SCHEMA CURATED;

In [ ]:
# Load curated data into local pandas for ML calculations
from datetime import timedelta

enriched_pdf = session.table('FACT_SALES_ENRICHED').to_pandas()
daily_pdf = session.table('DAILY_SALES_TREND').to_pandas()

print(f"Loaded FACT_SALES_ENRICHED: {len(enriched_pdf)} rows")
print(f"Loaded DAILY_SALES_TREND:   {len(daily_pdf)} rows")
print(f"Date range: {enriched_pdf['ORDER_DATE'].min()} to {enriched_pdf['ORDER_DATE'].max()}")
print(f"Total revenue: ${enriched_pdf['SALES_AMOUNT'].sum():,.2f}")

---
### ML Technique 1: Sales Forecasting
**Method:** Weighted Exponential Smoothing (WES, α=0.3)

**Business value:** Predicts next 7 days of revenue so the BI team can plan inventory and staffing. Includes confidence intervals for risk assessment.

**How it works:** WES gives 30% weight to the latest observation, 70% to the previous forecast. This adapts to trends while filtering noise.

In [ ]:
# Sort daily data by date
daily_sorted = daily_pdf.sort_values('DATE').reset_index(drop=True)
revenues = daily_sorted['REVENUE'].values

# Simple Moving Average baselines
daily_sorted['SMA_3'] = daily_sorted['REVENUE'].rolling(window=3, min_periods=1).mean()
daily_sorted['SMA_7'] = daily_sorted['REVENUE'].rolling(window=7, min_periods=1).mean()

# Weighted Exponential Smoothing (alpha = 0.3)
alpha = 0.3
wes = [revenues[0]]
for i in range(1, len(revenues)):
    wes.append(alpha * revenues[i] + (1 - alpha) * wes[-1])
daily_sorted['WES_FORECAST'] = wes

# Model accuracy
actual = revenues[3:]
predicted_sma3 = daily_sorted['SMA_3'].values[3:]
predicted_wes = np.array(wes[3:])

mae_sma3 = np.mean(np.abs(actual - predicted_sma3))
mape_sma3 = np.mean(np.abs((actual - predicted_sma3) / actual)) * 100
mae_wes = np.mean(np.abs(actual - predicted_wes))
mape_wes = np.mean(np.abs((actual - predicted_wes) / actual)) * 100

print("SALES FORECASTING — Model Performance")
print(f"  3-Day SMA:   MAE = ${mae_sma3:,.0f}   MAPE = {mape_sma3:.1f}%")
print(f"  WES (α=0.3): MAE = ${mae_wes:,.0f}   MAPE = {mape_wes:.1f}%")

In [ ]:
# Forecast next 7 days using mean-reverting WES
mean_rev = daily_sorted['REVENUE'].mean()
wes_last = wes[-1]
last_date = pd.to_datetime(daily_sorted['DATE'].max())

dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_factor = [0.95, 1.05, 1.00, 1.10, 1.00, 0.70, 0.65]

forecast_rows = []
for i in range(1, 8):
    fdate = last_date + timedelta(days=i)
    dow = fdate.weekday()
    weight = i / 7
    base = wes_last * (1 - weight) + mean_rev * weight
    forecast_val = round(base * dow_factor[dow], 2)
    forecast_rows.append({
        'FORECAST_DATE': fdate.strftime('%Y-%m-%d'),
        'DAY_OF_WEEK': dow_labels[dow],
        'FORECAST_REVENUE': forecast_val,
        'CONFIDENCE_LOWER': round(forecast_val * 0.75, 2),
        'CONFIDENCE_UPPER': round(forecast_val * 1.25, 2),
    })

forecast_pdf = pd.DataFrame(forecast_rows)
total_forecast = forecast_pdf['FORECAST_REVENUE'].sum()

print("\n7-Day Revenue Forecast:")
for _, row in forecast_pdf.iterrows():
    print(f"  {row['FORECAST_DATE']} ({row['DAY_OF_WEEK']}): ${row['FORECAST_REVENUE']:>10,.2f}  [{row['CONFIDENCE_LOWER']:>,.0f} – {row['CONFIDENCE_UPPER']:>,.0f}]")
print(f"  Total: ${total_forecast:,.2f}")

In [ ]:
# Save forecast to Snowflake
forecast_sdf = session.create_dataframe(forecast_pdf)
forecast_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.SALES_FORECAST', mode='overwrite')
print("✓ Saved CURATED.SALES_FORECAST")

---
### ML Technique 2: Customer Segmentation (RFM Analysis)
**Method:** Recency-Frequency-Monetary scoring with quartile-based segmentation

**Business value:** Classifies customers into actionable segments:
- **Champions** — High-value repeat buyers → reward & retain
- **Loyal Customers** — Consistent buyers → upsell opportunities
- **New Customers** — Recent first-time buyers → nurture campaigns
- **At Risk** — Haven't bought recently → win-back campaigns
- **Needs Attention** — Low engagement → evaluate marketing ROI

In [ ]:
# Build RFM scores
enriched_pdf['ORDER_DATE'] = pd.to_datetime(enriched_pdf['ORDER_DATE'])
reference_date = enriched_pdf['ORDER_DATE'].max() + timedelta(days=1)

rfm = enriched_pdf.groupby('CUSTOMER_KEY').agg(
    recency=('ORDER_DATE', lambda x: (reference_date - x.max()).days),
    frequency=('ORDER_NUMBER', 'count'),
    monetary=('SALES_AMOUNT', 'sum'),
).reset_index()

rfm['R_SCORE'] = pd.qcut(rfm['recency'], q=4, labels=[4,3,2,1]).astype(int)
rfm['F_SCORE'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['M_SCORE'] = pd.qcut(rfm['monetary'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['RFM_SCORE'] = rfm['R_SCORE'] + rfm['F_SCORE'] + rfm['M_SCORE']

def assign_segment(row):
    if row['RFM_SCORE'] >= 10: return 'Champions'
    elif row['RFM_SCORE'] >= 8: return 'Loyal Customers'
    elif row['R_SCORE'] >= 3 and row['F_SCORE'] <= 2: return 'New Customers'
    elif row['R_SCORE'] <= 2 and row['F_SCORE'] >= 2: return 'At Risk'
    elif row['RFM_SCORE'] <= 5: return 'Needs Attention'
    else: return 'Potential Loyalists'

rfm['SEGMENT'] = rfm.apply(assign_segment, axis=1)

seg_summary = rfm.groupby('SEGMENT').agg(
    CUSTOMER_COUNT=('CUSTOMER_KEY','count'),
    AVG_REVENUE=('monetary','mean'),
    TOTAL_REVENUE=('monetary','sum'),
).reset_index().sort_values('TOTAL_REVENUE', ascending=False)

print("CUSTOMER SEGMENTATION — RFM Analysis")
for _, r in seg_summary.iterrows():
    print(f"  {r['SEGMENT']:<22} {r['CUSTOMER_COUNT']:>4} customers  ${r['TOTAL_REVENUE']:>10,.0f} revenue")

In [ ]:
# Save RFM to Snowflake
rfm_out = rfm.rename(columns={'recency':'RECENCY_DAYS','frequency':'FREQUENCY','monetary':'MONETARY_VALUE'})
rfm_sdf = session.create_dataframe(rfm_out)
rfm_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.CUSTOMER_RFM', mode='overwrite')
print(f"✓ Saved CURATED.CUSTOMER_RFM ({len(rfm)} customers)")

seg_sdf = session.create_dataframe(seg_summary)
seg_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.SEGMENT_SUMMARY', mode='overwrite')
print(f"✓ Saved CURATED.SEGMENT_SUMMARY ({len(seg_summary)} segments)")

---
### ML Technique 3: Anomaly Detection
**Method:** Z-Score statistical analysis (threshold: |z| > 1.5)

**Business value:** Automatically flags days with unusually high or low sales. High anomalies = successful promotions or bulk orders. Low anomalies = system issues or market drops.

In [ ]:
# Z-Score anomaly detection
mean_revenue = daily_sorted['REVENUE'].mean()
std_revenue = daily_sorted['REVENUE'].std()

daily_sorted['Z_SCORE'] = ((daily_sorted['REVENUE'] - mean_revenue) / std_revenue).round(4)
daily_sorted['IS_ANOMALY'] = daily_sorted['Z_SCORE'].abs() > 1.5
daily_sorted['ANOMALY_TYPE'] = daily_sorted.apply(
    lambda r: 'HIGH' if r['Z_SCORE'] > 1.5 else ('LOW' if r['Z_SCORE'] < -1.5 else 'NORMAL'), axis=1
)

anomalies = daily_sorted[daily_sorted['IS_ANOMALY']]

print(f"ANOMALY DETECTION — Z-Score Analysis")
print(f"  Mean daily revenue: ${mean_revenue:,.2f}")
print(f"  Normal range: ${mean_revenue - 1.5*std_revenue:,.0f} – ${mean_revenue + 1.5*std_revenue:,.0f}")
print(f"  Detected {len(anomalies)} anomalous day(s):")
for _, r in anomalies.iterrows():
    print(f"    {str(r['DATE'])[:10]}  ${r['REVENUE']:>10,.2f}  z={r['Z_SCORE']:>+6.2f}  [{r['ANOMALY_TYPE']}]")

In [ ]:
# Save anomaly data
anomaly_out = daily_sorted[['DATE','REVENUE','PROFIT','ORDERS','SMA_3','SMA_7','WES_FORECAST','Z_SCORE','IS_ANOMALY','ANOMALY_TYPE']].copy()
anomaly_out['DATE'] = anomaly_out['DATE'].astype(str)
anomaly_sdf = session.create_dataframe(anomaly_out)
anomaly_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.DAILY_WITH_ANOMALIES', mode='overwrite')
print(f"✓ Saved CURATED.DAILY_WITH_ANOMALIES ({len(anomaly_out)} rows, {len(anomalies)} flagged)")

---
### ML Technique 4: Product Classification (BCG Matrix)
**Method:** Volume × Margin classification

**Business value:**
- **Stars** — High volume + high margin → invest and grow
- **Cash Cows** — High volume + lower margin → maintain, optimize costs
- **Question Marks** — Low volume + high margin → investigate growth
- **Dogs** — Low volume + low margin → consider discontinuing

In [ ]:
# Product profitability analysis
product_analysis = enriched_pdf.groupby(['PRODUCT_NAME','MODEL']).agg(
    REVENUE=('SALES_AMOUNT','sum'), PROFIT=('GROSS_PROFIT','sum'),
    ORDERS=('ORDER_NUMBER','count'), AVG_MARGIN=('PROFIT_MARGIN_PCT','mean'),
).reset_index()
product_analysis['PROFIT_PER_UNIT'] = (product_analysis['PROFIT'] / product_analysis['ORDERS']).round(2)

median_orders = product_analysis['ORDERS'].median()
median_margin = product_analysis['AVG_MARGIN'].median()

def classify_bcg(row):
    hv = row['ORDERS'] >= median_orders
    hm = row['AVG_MARGIN'] >= median_margin
    if hv and hm: return 'Star'
    elif hv and not hm: return 'Cash Cow'
    elif not hv and hm: return 'Question Mark'
    else: return 'Dog'

product_analysis['BCG_CLASS'] = product_analysis.apply(classify_bcg, axis=1)
product_analysis = product_analysis.sort_values('REVENUE', ascending=False)

print("PRODUCT CLASSIFICATION — BCG Matrix")
for _, r in product_analysis.iterrows():
    print(f"  {r['PRODUCT_NAME'][:28]:<30} {r['BCG_CLASS']:<15} {r['AVG_MARGIN']:>5.1f}%  {r['ORDERS']:>3} orders  ${r['PROFIT_PER_UNIT']:>8,.2f}/unit")

In [ ]:
product_sdf = session.create_dataframe(product_analysis)
product_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.PRODUCT_CLASSIFICATION', mode='overwrite')
print(f"✓ Saved CURATED.PRODUCT_CLASSIFICATION ({len(product_analysis)} products)")

---
### ML Technique 5: Territory Performance Scoring
**Method:** Composite weighted score (Revenue 35% + Profit 30% + Volume 20% + AOV 15%)

**Business value:** Ranks territories with actionable recommendations:
- **Invest Heavily** (≥70) → Expand team, increase marketing
- **Maintain & Grow** (40-70) → Steady investment
- **Evaluate** (20-40) → Analyze ROI
- **Optimize Costs** (<20) → Reduce overhead

In [ ]:
# Territory performance scoring
terr = enriched_pdf.groupby('TERRITORY_NAME').agg(
    REVENUE=('SALES_AMOUNT','sum'), PROFIT=('GROSS_PROFIT','sum'),
    ORDERS=('ORDER_NUMBER','count'), AVG_ORDER=('SALES_AMOUNT','mean'),
    UNIQUE_CUSTOMERS=('CUSTOMER_KEY','nunique'),
).reset_index()

for col in ['REVENUE','PROFIT','ORDERS','AVG_ORDER']:
    mn, mx = terr[col].min(), terr[col].max()
    rng = mx - mn if mx != mn else 1
    terr[f'{col}_SCORE'] = ((terr[col] - mn) / rng * 100).round(1)

terr['COMPOSITE_SCORE'] = (
    terr['REVENUE_SCORE']*0.35 + terr['PROFIT_SCORE']*0.30 +
    terr['ORDERS_SCORE']*0.20 + terr['AVG_ORDER_SCORE']*0.15
).round(1)

def recommend(s):
    if s >= 70: return 'Invest Heavily'
    elif s >= 40: return 'Maintain & Grow'
    elif s >= 20: return 'Evaluate'
    else: return 'Optimize Costs'

terr['RECOMMENDATION'] = terr['COMPOSITE_SCORE'].apply(recommend)
terr = terr.sort_values('COMPOSITE_SCORE', ascending=False)

print("TERRITORY SCORING — Investment Priorities")
for _, r in terr.iterrows():
    print(f"  {r['TERRITORY_NAME']:<18} Score: {r['COMPOSITE_SCORE']:>5.1f}  ${r['REVENUE']:>10,.0f}  → {r['RECOMMENDATION']}")

In [ ]:
terr_out = terr[['TERRITORY_NAME','REVENUE','PROFIT','ORDERS','AVG_ORDER','UNIQUE_CUSTOMERS','COMPOSITE_SCORE','RECOMMENDATION']]
terr_sdf = session.create_dataframe(terr_out)
terr_sdf.write.save_as_table('SALES_ANALYTICS_2025.CURATED.TERRITORY_SCORING', mode='overwrite')
print(f"✓ Saved CURATED.TERRITORY_SCORING ({len(terr_out)} territories)")

---
### Final Validation

In [ ]:
%%sql -r dataframe_2
-- Show all curated tables including ML results
SELECT TABLE_NAME, ROW_COUNT
FROM SALES_ANALYTICS_2025.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'CURATED'
ORDER BY TABLE_NAME;

In [ ]:
print("=" * 55)
print("  STEP 3 COMPLETE: DATA SCIENCE & ML")
print("=" * 55)
print(f"")
print(f"  Technique 1: Sales Forecasting")
print(f"    → 7-day total: ${total_forecast:,.2f}")
print(f"    → CURATED.SALES_FORECAST")
print(f"")
print(f"  Technique 2: Customer Segmentation (RFM)")
print(f"    → {len(rfm)} customers, {rfm['SEGMENT'].nunique()} segments")
print(f"    → CURATED.CUSTOMER_RFM")
print(f"")
print(f"  Technique 3: Anomaly Detection")
print(f"    → {len(anomalies)} anomalies flagged")
print(f"    → CURATED.DAILY_WITH_ANOMALIES")
print(f"")
print(f"  Technique 4: Product Classification (BCG)")
print(f"    → Stars: {len(product_analysis[product_analysis.BCG_CLASS=='Star'])}")
print(f"    → Cash Cows: {len(product_analysis[product_analysis.BCG_CLASS=='Cash Cow'])}")
print(f"    → CURATED.PRODUCT_CLASSIFICATION")
print(f"")
print(f"  Technique 5: Territory Scoring")
print(f"    → #1: {terr.iloc[0]['TERRITORY_NAME']} ({terr.iloc[0]['COMPOSITE_SCORE']})")
print(f"    → CURATED.TERRITORY_SCORING")
print(f"")
print(f"  ▶ NEXT: Run Step4_Delivery_CortexAnalyst.ipynb")
print(f"    Set up the chat interface for Rajendhar!")